# Registros afectados por variable

El propósito de este notebook es identificar la cantidad de registros afectados por cada variable editada, para completar la columna de registros afectados de la tabla ubicada en /docs/Registro_de_Transformaciones.md. Para lograrlo se compara el dataset crudo contra el dataset limpio, cruzando ambos por codigo, y se cuenta en cuántos registros cambió el valor de cada variable. En el caso de NIVEL, al ser una columna que se eliminó por completo, se reporta el total de filas.

In [3]:
import sys
from pathlib import Path

import pandas as pd

# Ubica la raiz del repo (funciona desde notebooks/ o desde la raiz) e importa config.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
import config

crudo = pd.concat(
    [pd.read_csv(archivo, dtype=str) for archivo in sorted((RAIZ / "data" / "raw").glob("diversificado_*.csv"))],
    ignore_index=True,
)
limpio = pd.read_csv(config.RUTA_FINAL, dtype=str)

# El crudo usa los nombres originales; se renombran a los del dataset limpio
# para poder comparar variable por variable con los mismos nombres.
crudo = crudo.rename(columns=config.RENOMBRES)
comparacion = crudo.merge(limpio, on="codigo", suffixes=("_crudo", "_limpio"))

# Variables que se modificaron durante la limpieza, en el orden del pipeline.
variables_editadas = [
    "director", "direccion", "area", "establecimiento", "supervisor",
    "telefono", "distrito", "plan", "direccion_departamental", "nivel",
]


def registros_afectados(variable):
    # NIVEL se elimino por completo, por lo que aplica a todas las filas.
    if variable not in limpio.columns:
        return len(limpio)
    antes = comparacion[f"{variable}_crudo"]
    despues = comparacion[f"{variable}_limpio"]
    # Un registro cambio si el valor difiere, tratando NaN == NaN como igual.
    iguales = (antes == despues) | (antes.isna() & despues.isna())
    return int((~iguales).sum())


resumen = pd.DataFrame({
    "variable": variables_editadas,
    "registros_afectados": [registros_afectados(variable) for variable in variables_editadas],
})
resumen

,variable,registros_afectados
0,director,61
1,direccion,5
2,area,3
3,establecimiento,5
4,supervisor,3
5,telefono,201
6,distrito,70
7,plan,8777
8,direccion_departamental,2025
9,nivel,11867
